In [ ]:
import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.express as px

from kebab.utils.dataset.t_rex.entity_fragment import EntityFragment
from kebab.utils.dataset.t_rex.t_rex_dataset_builder import TRexDatasetBuilder

Load the data

In [ ]:
# load the Wikidata person names dataset
dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "T-REx"
    / "Entities"
    / "2024-06-27"
    / "full"
    / "t_rex_entities_dataset.jsonl"
)

all_fragments = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fragment = EntityFragment.from_json(line.strip())
        all_fragments.append(fragment)

In [ ]:
print(f"Number of fragments: {len(all_fragments):,d}")
print("Sample fragment:")
all_fragments[0]

Filter by Type (optional)
---

In [ ]:
type_id = "Q154954"  # human
# type_id = "Q124250988"  # urban settlement
# type_id = "Q4830453"  # business

hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2024-06-27"
    / "wikidata_type_hierarchy.jsonl"
)

import json

graph = {}
with open(hierarchy_path, encoding="utf-8") as f:
    for line in f:
        node = json.loads(line.strip())
        graph[node["id"]] = node

types = TRexDatasetBuilder.collect_all_subtypes(graph, type_id)
print(f"Number of types: {len(types):,d}")

t_fragments = [f for f in all_fragments if any(t for t in f.entity_types if t in types)]
print(f"Number of fragments: {len(t_fragments):,d}")

In [ ]:
# fragments = all_fragments
fragments = t_fragments

Merge into entities

In [ ]:
entity_to_fragments = defaultdict(list)
for fragment in fragments:
    entity_to_fragments[fragment.entity_id].append(fragment)

# merge
entities = {entity_id: EntityFragment.merge(fragments) for entity_id, fragments in entity_to_fragments.items()}

Convert to DataFrame

In [ ]:
# convert fragments to a DataFrame
df = pd.DataFrame(fragments)
df = df.drop(columns=["source_ids", "evidence_map"])

# add counts
df["names"] = df["properties"].apply(lambda x: x["names"] if "names" in x else [])
df["names_count"] = df["names"].apply(len)
df["properties_count"] = df["properties"].apply(len)
df["property_values_count"] = df["properties"].apply(lambda x: sum(len(v) for v in x.values()))

df.shape

In [ ]:
edf = pd.DataFrame(entities.values())
edf = edf.drop(columns=["source_ids", "evidence_map"])

# add counts
edf["names"] = edf["properties"].apply(lambda x: x["names"] if "names" in x else [])
edf["names_count"] = edf["names"].apply(len)
edf["properties_count"] = edf["properties"].apply(len)
edf["property_values_count"] = edf["properties"].apply(lambda x: sum(len(v) for v in x.values()))

edf.shape

Fragment statistics
---

In [ ]:
print(f"Number of fragments: {len(df):,d}")
print(f"Average number of names per fragment: {df['names_count'].mean():.2f}")
print(f"Average number of properties per fragment: {df['properties_count'].mean():.2f}")
print(f"Average number of property values per fragment: {df['property_values_count'].mean():.2f}")

In [ ]:
# histogram
take = 10
fig = px.histogram(
    x=df["properties_count"].clip(upper=take),
    nbins=2 * take,
    title="Fragments by number of Properties",
    labels={"x": "number of properties", "y": "number of fragments"},
)

total = len(df)
groups = sorted(df["properties_count"].unique())[:take]
for i in groups:
    count = (df["properties_count"] == i).sum() if i != groups[-1] else (df["properties_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total:.1%}", showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="fragment count")
fig.update_layout(width=1000)
fig.show()

Entity statistics
---

In [ ]:
print(f"Number of entities: {df['entity_id'].nunique():,d}")
gdf = df.groupby("entity_id")
print(f"Average number of fragments per entity: {gdf.size().mean():.2f}")
print(f"Entities with at least 2 fragments: {gdf.size().ge(2).sum():,d}")
print(f"Entities with at least 5 fragments: {gdf.size().ge(5).sum():,d}")

In [ ]:
# histogram
take = 10
fig = px.histogram(
    x=gdf.size().clip(upper=take),
    nbins=2 * take,
    title="Entities by number of Fragments",
    labels={"x": "number of fragments", "y": "number of entities"},
)

total = len(gdf)
groups = sorted(gdf.size().unique())[:take]
for i in groups:
    count = (gdf.size() == i).sum() if i != groups[-1] else (gdf.size() >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total:.1%}", showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="entity count")
fig.update_layout(width=1000)
fig.show()

Properties
---
(at the fragment level)

In [ ]:
# property occurrences
prop_counts = {}
for row in df["properties"]:
    for k in row.keys():
        prop_counts[k] = prop_counts.get(k, 0) + 1

prop_counts = pd.Series(prop_counts)
prop_counts = prop_counts.sort_values(ascending=False)
prop_counts = prop_counts.rename("count")
prop_counts[:50]

In [ ]:
# plot counts
take = 20
prop_counts = prop_counts[~prop_counts.index.str.startswith("names")]

fig = px.bar(
    x=prop_counts[:take].index,
    y=prop_counts[:take].values,
    title="Top properties by occurrence",
    labels={"x": "property", "y": "count"},
)

total = len(df)
for i, value in zip(prop_counts[:take].index, prop_counts[:take].values):
    fig.add_annotation(x=i, y=value, text=f"{value / total:.1%}", showarrow=False, yshift=10)

fig.update_layout(width=1000)
fig.show()

Properties
---
(at the entity level)

In [ ]:
# property occurrences
prop_counts = {}
for row in edf["properties"]:
    for k in row.keys():
        prop_counts[k] = prop_counts.get(k, 0) + 1

prop_counts = pd.Series(prop_counts)
prop_counts = prop_counts.sort_values(ascending=False)
prop_counts = prop_counts.rename("count")
prop_counts[:50]

In [ ]:
# plot counts
take = 20
prop_counts = prop_counts[~prop_counts.index.str.startswith("names")]

fig = px.bar(
    x=prop_counts[:take].index,
    y=prop_counts[:take].values,
    title="Top properties by occurrence",
    labels={"x": "property", "y": "count"},
)

total = len(edf)
for i, value in zip(prop_counts[:take].index, prop_counts[:take].values):
    fig.add_annotation(x=i, y=value, text=f"{value / total:.1%}", showarrow=False, yshift=10)

fig.update_layout(width=1000)
fig.show()

Fragments with the most properties
---

In [ ]:
# top fragments by the number of properties
df.sort_values("properties_count", ascending=False).head(10)

Fragments with the most names
---

In [ ]:
# top fragments by the number of names
df.sort_values("names_count", ascending=False).head(50)

Entities with the most fragments
---

In [ ]:
# top entities by the number of fragments
gdf.agg(names=("names", "first"), fragments_count=("entity_id", "size")).sort_values(
    "fragments_count", ascending=False
).head(10)

Random sample
---

In [ ]:
# random sample
df.sample(10)

Properties overlap
---

In [ ]:
# for each property how often that two distinct entities have the same value; do that too for whether two entities have a value for this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity in entities.values():
    for prop_name, values in entity.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(entities)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_probs = arr / entity_count
    prob = (ent_probs**2).sum()
    ent_val_count = entity_value_counts[prop_name]
    rows.append((prop_name, len(value_counts), prob, ent_val_count / entity_count))

overlap_df = pd.DataFrame(rows, columns=["property", "distinct_value_count", "overlap_prob", "entities_fraction"])
overlap_df = overlap_df.sort_values("entities_fraction", ascending=False)
overlap_df.head(100)